## M2 MobileNet OOD
The purpose of this notebook will be to try and change the MobileNet head to transfer learn it from ImageNet to CIFAR10 dataset. The aim is to implement a simple out-of-distribution (OOD) detection method using many different techniques and then test if it actually does something

<div style="text-align: center;">
<img src="https://www.researchgate.net/publication/376752661/figure/fig3/AS:11431281213995739@1703264754673/MobileNet-V1-architecture.png" width="900" alt="Mobile Net Architecture">
</div>

## 1. Setup
Import necessary libraries and set up the environment for training and evaluating the MobileNet model on the CIFAR10 dataset.

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torchvision.models import mobilenet_v3_small, MobileNet_V3_Small_Weights
from torch.utils.data import DataLoader
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm

### 1.1 Device Configuration
Set up the device for training (GPU if available, otherwise CPU).

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cpu


### 1.2 Getting training and testing dataset
Because MobileNet is pretrained on ImageNet, we need to resize the CIFAR10 images from 32x32 to 224x224 and normalize them using ImageNet's mean and std.

In [3]:
transform = transforms.Compose([
    transforms.Resize(224),  # MobileNet expects 224x224 images
    transforms.ToTensor(),  # Convert PIL image to tensor
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])  # ImageNet normalization
])

train_dataset_id = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transform)
test_dataset_id = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=transform)

100%|██████████| 170M/170M [00:08<00:00, 21.1MB/s] 


In [4]:
print(f"Number of training samples: {len(train_dataset_id)}")
print(f"Number of test samples: {len(test_dataset_id)}")

Number of training samples: 50000
Number of test samples: 10000


### 1.3 Getting the OOD dataset
For OOD detection, we will use the SVHN dataset as our out-of-distribution data

In [ ]:
ood_dataset_svhn = torchvision.datasets.SVHN(root='./data', split='test', download=True, transform=transform)
ood_loader_svhn = DataLoader(ood_dataset_svhn, batch_size=64, shuffle=False, num_workers=2)
print(f"Amount of SVHN samples (OOD - SVHN): {len(ood_dataset_svhn)}")

### 1.4 Getting the model
For our test we will use MobileNet V3 Small pretrained on ImageNet

We will need to do the following:
- Load the pretrained MobileNet V3 Small model
- Freeze the feature extractor layers
- Replace the classifier head to output 10 classes for CIFAR-10

<div style="text-align: center;">
<img src="https://www.nim.org/fileadmin/PUBLIC/2_NIM_Forschung/mobilelogoshootout.png" width="900" alt="MobileNet vs Yolo">
</div>

In [ ]:
model = mobilenet_v3_small(weights=MobileNet_V3_Small_Weights.IMAGENET1K_V1)

In [ ]:
for param in model.parameters():
    param.requires_grad = False

In [ ]:
num_features = model.classifier[-1].in_features # Get the number of input features to the last layer
model.classifier[-1] = nn.Linear(num_features, 10) # Create a new linear layer with 10 outputs for CIFAR-10 and num_features inputs

In [ ]:
model = model.to(device)